# English-to-French Neural Machine Translation

In this assignment, you will fine-tune pre-trained MarianMT models for English-to-French translation.
Unlike the text classification task, this is a **sequence-to-sequence** problem: the model takes
an English sentence as input and generates a full French sentence as output.

You will:
- Compare a small vs. large MarianMT model (Task 1)
- Tune hyperparameters to improve BLEU score (Task 2)
- Run your best model on a held-out test set and perform error analysis (Task 3)

**Evaluation metric:** BLEU score (via `sacrebleu`). Higher is better. Scores above 25 are reasonable for En-Fr.

In [ ]:
# Install required packages
# Note: remove this cell if running locally and packages are already installed
!pip install transformers datasets sacrebleu sentencepiece sacremoses wandb -q

In [ ]:
# Standard imports
import torch
import numpy as np
import json
import os
import random
from torch import nn
from torch.utils.data import DataLoader
from transformers import MarianMTModel, MarianTokenizer
from datasets import load_dataset, Dataset
import sacrebleu
import wandb

# Fix random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [ ]:
# Set your compute device
# Colab with Nvidia GPU:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Apple Silicon:
# device = 'mps' if torch.backends.mps.is_available() else 'cpu'

print(f"Using device: {device}")

## Utility Functions

These functions handle data loading, tokenization, training, evaluation, and BLEU scoring.
Implement the parts marked with `# TODO`.

In [ ]:
def load_data(max_train_samples=50000, max_val_samples=2000):
    """
    Load the OPUS Books English-French dataset from Hugging Face.
    We subsample the training set to keep training times manageable on Colab.

    Args:
        max_train_samples: Maximum number of training examples to use
        max_val_samples: Maximum number of validation examples to use

    Returns:
        train_dataset, val_dataset: Hugging Face Dataset objects with 'en' and 'fr' fields
    """
    # TODO: Load the dataset using load_dataset
    # Dataset name: "Helsinki-NLP/opus_books"
    # Language pair name argument: "en-fr"
    # This dataset only has a 'train' split, so you will need to manually split it.
    # Hint: use dataset['train'].train_test_split(test_size=...) to create a val split
    #
    # Each example looks like: {'translation': {'en': '...', 'fr': '...'}}
    # Flatten each example so it has 'en' and 'fr' keys directly.
    # Subsample to max_train_samples and max_val_samples for speed.

    raise NotImplementedError("TODO: Implement load_data")

    return train_dataset, val_dataset

In [ ]:
def get_tokenizer(model_name):
    """
    Load and return the MarianTokenizer for the given model name.

    Args:
        model_name: HuggingFace model name (e.g. 'Helsinki-NLP/opus-mt-en-fr')

    Returns:
        tokenizer: MarianTokenizer instance
    """
    # TODO: Load and return the MarianTokenizer using MarianTokenizer.from_pretrained
    raise NotImplementedError("TODO: Implement get_tokenizer")

In [ ]:
def tokenize_batch(batch, tokenizer, max_length=128):
    """
    Tokenize a batch of (English, French) sentence pairs.

    For seq2seq models, we tokenize the source (English) as the model inputs
    and the target (French) as the labels. The model internally shifts the
    labels right to create decoder inputs.

    Args:
        batch: List of dicts with 'en' and 'fr' keys
        tokenizer: MarianTokenizer
        max_length: Maximum token sequence length

    Returns:
        Dict with 'input_ids', 'attention_mask', and 'labels' tensors
    """
    # TODO: Implement tokenization for seq2seq
    # 1. Extract source (English) sentences from batch
    # 2. Extract target (French) sentences from batch
    # 3. Tokenize sources and targets together using the tokenizer's text= and text_target=
    #    arguments in a single call:
    #      tokenizer(text=sources, text_target=targets, padding=True, truncation=True,
    #                max_length=max_length, return_tensors='pt')
    #    Why text_target=? The big model (opus-mt-tc-big-en-fr) uses separate encoder and
    #    decoder vocabularies. Passing French through text_target= ensures the tokenizer
    #    uses the TARGET vocabulary for French — if you tokenize French with a plain second
    #    call instead, you may get token IDs from the wrong vocab, causing NaN losses.
    # 4. The returned object has 'input_ids', 'attention_mask', and 'labels' keys.
    #    Replace padding token ids in labels with -100 so they are ignored in the loss:
    #    Hint: labels[labels == tokenizer.pad_token_id] = -100
    # 5. Return a dict with 'input_ids', 'attention_mask', and 'labels'

    raise NotImplementedError("TODO: Implement tokenize_batch")

In [ ]:
def compute_bleu(predictions, references):
    """
    Compute corpus-level BLEU score using sacrebleu.

    Args:
        predictions: List of predicted (translated) strings
        references: List of reference (ground truth) strings

    Returns:
        bleu_score: Float BLEU score (0-100 scale)
    """
    # TODO: Use sacrebleu.corpus_bleu to compute the BLEU score
    # Note: sacrebleu expects references as a list of lists: [references_list]
    # Return the .score attribute (a float)
    raise NotImplementedError("TODO: Implement compute_bleu")

In [ ]:
def generate_translations(model, tokenizer, dataset, batch_size=32, device='cuda',
                          num_beams=4, max_length=128):
    """
    Generate translations for a dataset using beam search.

    Args:
        model: Trained MarianMTModel
        tokenizer: MarianTokenizer
        dataset: Dataset with 'en' and 'fr' fields
        batch_size: Number of examples per batch during generation
        device: Compute device
        num_beams: Beam search width (1 = greedy decoding)
        max_length: Maximum output sequence length

    Returns:
        predictions: List of translated strings
        references: List of reference strings
    """
    # TODO: Implement generation
    # 1. Set model to eval mode and move to device
    # 2. Loop through the dataset in batches
    # 3. For each batch:
    #    a. Tokenize the English sources
    #    b. Call model.generate() with num_beams and max_length
    #       Hint: model.generate(**inputs, num_beams=num_beams, max_length=max_length)
    #    c. Decode the generated token ids using tokenizer.batch_decode
    #       with skip_special_tokens=True
    # 4. Collect all predictions and corresponding French references
    # 5. Return (predictions, references)

    raise NotImplementedError("TODO: Implement generate_translations")

In [ ]:
def init_wandb(config, project_name):
    """Initialize a wandb run with the given config dict."""
    wandb.init(project=project_name, config=config)
    return wandb.config

In [ ]:
def train(model, train_dataset, val_dataset, tokenizer,
          num_epochs=5, batch_size=16, lr=5e-5, weight_decay=1e-4,
          num_beams=4, label_smoothing=0.0, device='cuda', use_wandb=False):
    """
    Fine-tune a MarianMTModel on the translation task.

    For seq2seq models, the loss is computed internally when 'labels' are passed
    to model(**batch). You do NOT need a separate loss function — the model
    returns a .loss attribute directly.

    Args:
        model: MarianMTModel to fine-tune
        train_dataset: Training dataset
        val_dataset: Validation dataset
        tokenizer: MarianTokenizer
        num_epochs: Number of training epochs
        batch_size: Training batch size
        lr: Learning rate for AdamW
        weight_decay: Weight decay
        num_beams: Beam width used during validation generation
        label_smoothing: Label smoothing factor (0.0 = disabled)
        device: Compute device
        use_wandb: Whether to log metrics to wandb

    Returns:
        model: Trained model
        history: Dict with train_loss, val_loss, val_bleu lists
    """
    # TODO: Move model to device and set to train mode

    dataloader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda b: tokenize_batch(b, tokenizer)
    )

    # TODO: Initialize AdamW optimizer with lr and weight_decay

    history = {'train_loss': [], 'val_loss': [], 'val_bleu': []}
    global_step = 0

    for epoch in range(num_epochs):
        model.train()
        epoch_losses = []

        for batch in dataloader:
            # TODO: Move all batch tensors to device

            # TODO: Forward pass
            # Pass the batch to the model: outputs = model(**batch)
            # The model returns a Seq2SeqLMOutput; access outputs.loss for the loss

            # TODO: Backward pass and optimizer step
            # 1. Zero gradients
            # 2. loss.backward()
            # 3. (Recommended) Clip gradients to prevent exploding gradients:
            #    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            # 4. optimizer.step()
            #
            # Tip: If you encounter NaN losses, skip the offending batch:
            #    if torch.isnan(loss): print('NaN detected, skipping'); continue

            epoch_losses.append(loss.item())
            global_step += 1

            if global_step % 100 == 0:
                print(f"Epoch {epoch+1}, Step {global_step}, Loss: {epoch_losses[-1]:.4f}")
                if use_wandb:
                    wandb.log({'train_loss_step': epoch_losses[-1], 'global_step': global_step})

        epoch_train_loss = np.mean(epoch_losses)

        # Validation: compute val loss and BLEU
        # TODO: Call evaluate() to get val_loss and val_bleu
        # val_loss, val_bleu = evaluate(model, val_dataset, tokenizer, batch_size, num_beams, device)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(val_loss)
        history['val_bleu'].append(val_bleu)

        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val BLEU: {val_bleu:.2f}")

        if use_wandb:
            wandb.log({
                'epoch': epoch + 1,
                'train_loss': epoch_train_loss,
                'val_loss': val_loss,
                'val_bleu': val_bleu
            })

    return model, history

In [ ]:
@torch.no_grad()
def evaluate(model, dataset, tokenizer, batch_size=32, num_beams=4, device='cuda'):
    """
    Evaluate the model on a dataset, returning validation loss and BLEU score.

    Args:
        model: MarianMTModel
        dataset: Validation/test dataset with 'en' and 'fr' fields
        tokenizer: MarianTokenizer
        batch_size: Batch size for evaluation
        num_beams: Beam width for generation
        device: Compute device

    Returns:
        val_loss: Mean cross-entropy loss over the dataset
        val_bleu: Corpus BLEU score
    """
    # TODO: Set model to eval mode

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=lambda b: tokenize_batch(b, tokenizer)
    )

    # TODO: Compute validation loss
    # Loop through the dataloader, pass each batch to the model,
    # collect outputs.loss values, and compute the mean.

    # TODO: Generate translations using generate_translations() and compute BLEU
    # predictions, references = generate_translations(model, tokenizer, dataset, batch_size, device, num_beams)
    # val_bleu = compute_bleu(predictions, references)

    return val_loss, val_bleu

In [ ]:
def predict_on_test_set(model, tokenizer, test_file_path, output_file_path,
                        device='cuda', num_beams=4, max_length=128):
    """
    Generate translations for the held-out test set and save to file.

    Args:
        model: Trained MarianMTModel
        tokenizer: MarianTokenizer
        test_file_path: Path to test-DIST.json (list of {'en': ...} dicts)
        output_file_path: Path to write test-translations.txt (one translation per line)
        device: Compute device
        num_beams: Beam width for generation
        max_length: Maximum output length
    """
    with open(test_file_path, 'r', encoding='utf-8') as f:
        test_data = json.load(f)                          
    sentences = [item['en'] for item in test_data]     

    print(f"Loaded {len(sentences)} test sentences from {test_file_path}")

    # TODO: Set model to eval mode and move to device

    translations = []

    # TODO: Generate a translation for each sentence in sentences.
    # Tokenize each sentence, run model.generate(), and decode the output.
    # Append each decoded French string to translations.

    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(translations))                  

    print(f"Translations saved to {output_file_path}")

## Dataset Exploration

Before training, let's explore the dataset to understand its structure and characteristics.

In [ ]:
# Load dataset
train_dataset, val_dataset = load_data(max_train_samples=50000, max_val_samples=2000)
print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

In [ ]:
# Inspect some examples
print("Sample training examples:")
for i in range(5):
    ex = train_dataset[i]
    print(f"  EN: {ex['en']}")
    print(f"  FR: {ex['fr']}")
    print()

In [ ]:
# Explore tokenization differences between the two models
small_tokenizer = get_tokenizer("Helsinki-NLP/opus-mt-en-fr")
big_tokenizer   = get_tokenizer("Helsinki-NLP/opus-mt-tc-big-en-fr")

example_en = train_dataset[0]['en']
example_fr = train_dataset[0]['fr']

print(f"Source (EN): {example_en}")
print(f"Small model tokenization (EN): {small_tokenizer.tokenize(example_en)}")
print(f"Big   model tokenization (EN): {big_tokenizer.tokenize(example_en)}")
print()
print(f"Reference (FR): {example_fr}")
print(f"Small model tokenization (FR): {small_tokenizer.tokenize(example_fr)}")
print(f"Big   model tokenization (FR): {big_tokenizer.tokenize(example_fr)}")

In [ ]:
# Analyze sentence length distribution (source vs target)
import matplotlib.pyplot as plt

en_lengths = [len(ex['en'].split()) for ex in train_dataset]
fr_lengths = [len(ex['fr'].split()) for ex in train_dataset]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(en_lengths, bins=50, color='steelblue', alpha=0.7)
axes[0].set_title("English Sentence Length Distribution")
axes[0].set_xlabel("Number of words")
axes[0].set_ylabel("Count")

axes[1].hist(fr_lengths, bins=50, color='coral', alpha=0.7)
axes[1].set_title("French Sentence Length Distribution")
axes[1].set_xlabel("Number of words")

plt.tight_layout()
plt.savefig("length_distribution.png", dpi=150)
plt.show()
print(f"Avg EN length: {np.mean(en_lengths):.1f} words")
print(f"Avg FR length: {np.mean(fr_lengths):.1f} words")

## Task 1: Model Selection

Compare the **small** MarianMT model (`Helsinki-NLP/opus-mt-en-fr`) against the **large** model
(`Helsinki-NLP/opus-mt-tc-big-en-fr`) using the same baseline configuration.

Both models share the MarianMT encoder-decoder architecture, but differ in depth and hidden size.
You will evaluate them by validation BLEU score after each epoch of fine-tuning.

In [ ]:
def run_model_selection():
    """Run Task 1: compare small vs. large MarianMT model."""

    model_configs = [
        {
            "model_name":   "Helsinki-NLP/opus-mt-en-fr",
            "display_name": "Small (opus-mt-en-fr)"
        },
        {
            "model_name":   "Helsinki-NLP/opus-mt-tc-big-en-fr",
            "display_name": "Big (opus-mt-tc-big-en-fr)"
        },
    ]

    # Baseline training hyperparameters — do NOT change for Task 1
    baseline_params = {
        "num_epochs": 3,
        "batch_size": 16,
        "lr": 5e-5,
        "weight_decay": 1e-4,
        "num_beams": 4,
    }

    results = []

    for cfg in model_configs:
        print(f"\n{'='*55}")
        print(f"Training: {cfg['display_name']}")
        print(f"{'='*55}")

        tokenizer = get_tokenizer(cfg['model_name'])

        # TODO: Load the model using MarianMTModel.from_pretrained(cfg['model_name'])
        # Important: pass torch_dtype=torch.float32 when loading the model.
        # Without this, some hardware will load weights in float16, which causes
        # NaN losses during training — especially with the big model.
        model = None  # replace with your implementation

        # TODO: Print the number of trainable parameters
        num_params = None  # replace with your implementation
        print(f"Trainable parameters: {num_params:,}")

        wandb_cfg = {**cfg, **baseline_params}
        init_wandb(wandb_cfg, "nmt-task1-model-selection")

        model, history = train(
            model=model,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            tokenizer=tokenizer,
            **baseline_params,
            device=device,
            use_wandb=True
        )

        val_loss, val_bleu = evaluate(model, val_dataset, tokenizer,
                                      batch_size=32, num_beams=baseline_params['num_beams'],
                                      device=device)

        results.append({
            'display_name': cfg['display_name'],
            'model_name':   cfg['model_name'],
            'val_bleu':     val_bleu,
            'val_loss':     val_loss,
            'history':      history,
            'model':        model,
            'tokenizer':    tokenizer
        })

        wandb.finish()

    print("\n--- Task 1 Results ---")
    print(f"{'Model':<35} {'Val BLEU':>10} {'Val Loss':>10}")
    print("-" * 58)
    for r in results:
        print(f"{r['display_name']:<35} {r['val_bleu']:>10.2f} {r['val_loss']:>10.4f}")

    return results

In [ ]:
task1_results = run_model_selection()

In [ ]:
# TODO: Select the best model from Task 1
# 1. Find the result with the highest val_bleu
# 2. Extract best_model, best_model_name, best_tokenizer

best_result     = None  # TODO
best_model      = None  # TODO
best_model_name = None  # TODO
best_tokenizer  = None  # TODO

print(f"Selected model for Task 2: {best_model_name}")

In [ ]:
# Vocabulary Diagnostic — run this after selecting your best model
# This checks that your tokenizer and model are compatible.
# If you see an ERROR below, it likely means you are using text= instead of
# text_target= in tokenize_batch, which feeds French through the encoder vocab
# instead of the decoder vocab on models with separate vocabularies.
print(f"Model vocab size (config):  {best_model.config.vocab_size}")
print(f"Tokenizer vocab size:       {len(best_tokenizer)}")

test_batch = tokenize_batch(list(train_dataset.select(range(5))), best_tokenizer)
max_input_id = test_batch['input_ids'].max().item()
valid_labels = test_batch['labels'][test_batch['labels'] != -100]
max_label_id = valid_labels.max().item()
print(f"Max input token ID:  {max_input_id}")
print(f"Max label token ID:  {max_label_id}")

if max_input_id >= best_model.config.vocab_size or max_label_id >= best_model.config.vocab_size:
    print("ERROR: Token IDs exceed model embedding size — check tokenize_batch.")
else:
    print("OK: All token IDs are within valid range.")

## Task 2: Hyperparameter Tuning

Using the best model from Task 1, experiment with at least 5 configurations.
Track all runs in wandb and compare them using BLEU score on the validation set.

Parameters to tune:
- **Learning rate** (e.g. 5e-5, 2e-4, 5e-4)
- **Batch size** (e.g. 8, 16, 32)
- **Beam size** at inference (e.g. 1, 4, 8) — this affects quality without changing training
- **Freeze encoder** (True/False)
- **Label smoothing** (e.g. 0.0, 0.1)

In [ ]:
# Define your hyperparameter configurations — modify all except Baseline
hp_configs = [
    # Baseline — do NOT modify
    {
        "config_name":    "Baseline",
        "num_epochs":     5,
        "batch_size":     16,
        "lr":             5e-5,
        "weight_decay":   1e-4,
        "num_beams":      4,
        "freeze_encoder": False,
        "label_smoothing":0.0,
    },
    # Config 1 — modify this
    {
        "config_name":    "Config 1",
        "num_epochs":     5,
        "batch_size":     16,
        "lr":             2e-4,
        "weight_decay":   1e-4,
        "num_beams":      4,
        "freeze_encoder": False,
        "label_smoothing":0.0,
    },
    # Config 2 — modify this
    {
        "config_name":    "Config 2",
        "num_epochs":     5,
        "batch_size":     32,
        "lr":             5e-5,
        "weight_decay":   1e-4,
        "num_beams":      8,
        "freeze_encoder": False,
        "label_smoothing":0.0,
    },
    # Config 3 — modify this
    {
        "config_name":    "Config 3",
        "num_epochs":     5,
        "batch_size":     16,
        "lr":             5e-5,
        "weight_decay":   1e-4,
        "num_beams":      4,
        "freeze_encoder": True,
        "label_smoothing":0.0,
    },
    # Config 4 — modify this
    {
        "config_name":    "Config 4",
        "num_epochs":     5,
        "batch_size":     8,
        "lr":             2e-5,
        "weight_decay":   1e-4,
        "num_beams":      4,
        "freeze_encoder": False,
        "label_smoothing":0.1,
    },
    # Config 5 — modify this
    {
        "config_name":    "Config 5",
        "num_epochs":     5,
        "batch_size":     16,
        "lr":             5e-4,
        "weight_decay":   1e-4,
        "num_beams":      1,
        "freeze_encoder": False,
        "label_smoothing":0.0,
    },
]

In [ ]:
def freeze_encoder(model):
    """
    Freeze all parameters in the model's encoder.

    Args:
        model: MarianMTModel
    """
    # TODO: Iterate over model.model.encoder.parameters() and set requires_grad = False
    raise NotImplementedError("TODO: Implement freeze_encoder")

In [ ]:
def run_hyperparameter_tuning(model_name, tokenizer):
    """Run all hyperparameter configurations and track results."""

    print(f"\nHyperparameter tuning for: {model_name}")

    results = []
    best_val_bleu = 0
    best_model = None
    best_config_idx = 0

    for i, cfg in enumerate(hp_configs):
        print(f"\n--- Config {i+1}/{len(hp_configs)}: {cfg['config_name']} ---")

        # Load a fresh copy of the model for each config
        # TODO: Load model using MarianMTModel.from_pretrained(model_name)
        # Remember: pass torch_dtype=torch.float32 (same reason as Task 1)
        model = None  # replace

        # TODO: If cfg['freeze_encoder'] is True, call freeze_encoder(model)

        init_wandb({**cfg, 'model_name': model_name}, "nmt-task2-hyperparam-tuning")

        model, history = train(
            model=model,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            tokenizer=tokenizer,
            num_epochs=cfg['num_epochs'],
            batch_size=cfg['batch_size'],
            lr=cfg['lr'],
            weight_decay=cfg['weight_decay'],
            num_beams=cfg['num_beams'],
            label_smoothing=cfg['label_smoothing'],
            device=device,
            use_wandb=True
        )

        val_loss, val_bleu = evaluate(model, val_dataset, tokenizer,
                                      batch_size=32, num_beams=cfg['num_beams'],
                                      device=device)

        wandb.log({'final_val_bleu': val_bleu, 'final_val_loss': val_loss})
        wandb.finish()

        results.append({**cfg, 'val_loss': val_loss, 'val_bleu': val_bleu, 'model': model})

        if val_bleu > best_val_bleu:
            best_val_bleu  = val_bleu
            best_model     = model
            best_config_idx = i

    # Print summary table
    print("\n--- Task 2 Results ---")
    print(f"{'Config':<12} {'LR':>8} {'Batch':>6} {'Beams':>6} {'FrzEnc':>7} {'LblSmth':>8} {'Val BLEU':>10}")
    print("-" * 62)
    for r in results:
        print(f"{r['config_name']:<12} {r['lr']:>8.0e} {r['batch_size']:>6} "
              f"{r['num_beams']:>6} {str(r['freeze_encoder']):>7} "
              f"{r['label_smoothing']:>8.2f} {r['val_bleu']:>10.2f}")

    print(f"\nBest config: {results[best_config_idx]['config_name']} "
          f"| Val BLEU: {best_val_bleu:.2f}")

    return results, best_model, best_config_idx

In [ ]:
tuning_results, best_tuned_model, best_config_idx = run_hyperparameter_tuning(
    best_model_name, best_tokenizer
)

## Task 3: Final Run on the Test Set and Error Analysis

### Part 1: Generate test set translations

Run your best model on `test-DIST.json` and save translations to `test-translations.json`.

In [ ]:
# Generate translations on the held-out test set
predict_on_test_set(
    model=best_tuned_model,
    tokenizer=best_tokenizer,
    test_file_path="test-DIST.json",
    output_file_path="test-translations.txt",
    device=device,
    num_beams=hp_configs[best_config_idx]['num_beams']
)

### Part 2: Error Analysis

Analyze your model's translations on the **validation set** to understand failure modes.
Implement the cells below to:
1. Compute sentence-level BLEU for each validation example
2. Identify the worst and best translations
3. Categorize and discuss error patterns

In [ ]:
def compute_sentence_bleu(prediction, reference):
    """
    Compute sentence-level BLEU for a single translation pair.

    Args:
        prediction: Translated string
        reference: Reference string

    Returns:
        bleu: Float sentence BLEU score
    """
    # TODO: Use sacrebleu.sentence_bleu(prediction, [reference]).score
    raise NotImplementedError("TODO: Implement compute_sentence_bleu")

In [ ]:
# Generate translations on validation set for analysis
# TODO: Call generate_translations on the validation set to get predictions and references
val_predictions, val_references = None, None  # replace

# TODO: Compute sentence-level BLEU for each example
sentence_bleus = None  # replace — should be a list of floats

print(f"Corpus BLEU: {compute_bleu(val_predictions, val_references):.2f}")
print(f"Mean sentence BLEU: {np.mean(sentence_bleus):.2f}")

In [ ]:
# Find worst and best translations
n_samples = 25

# TODO: Get indices of the n_samples lowest sentence BLEU scores
worst_indices = None  # replace

# TODO: Get indices of the n_samples highest sentence BLEU scores
best_indices = None  # replace

print("=== WORST TRANSLATIONS (low BLEU) ===")
for idx in worst_indices:
    src = val_dataset[idx]['en']
    ref = val_references[idx]
    hyp = val_predictions[idx]
    bleu = sentence_bleus[idx]
    print(f"\nBLEU: {bleu:.2f}")
    print(f"  SRC: {src}")
    print(f"  REF: {ref}")
    print(f"  HYP: {hyp}")

print("\n=== BEST TRANSLATIONS (high BLEU) ===")
for idx in best_indices:
    src = val_dataset[idx]['en']
    ref = val_references[idx]
    hyp = val_predictions[idx]
    bleu = sentence_bleus[idx]
    print(f"\nBLEU: {bleu:.2f}")
    print(f"  SRC: {src}")
    print(f"  REF: {ref}")
    print(f"  HYP: {hyp}")

In [ ]:
# Analyze: sentence length vs. BLEU score
import matplotlib.pyplot as plt

src_lengths = [len(val_dataset[i]['en'].split()) for i in range(len(val_dataset))]

plt.figure(figsize=(8, 5))
plt.scatter(src_lengths, sentence_bleus, alpha=0.3, s=10, color='steelblue')
plt.xlabel("Source sentence length (words)")
plt.ylabel("Sentence BLEU")
plt.title("Sentence Length vs. Translation Quality")
plt.savefig("length_vs_bleu.png", dpi=150)
plt.show()

In [ ]:
# TODO: Write your error analysis discussion below as a markdown/comment cell.
#
# Your analysis should cover:
# 1. What error categories do you observe in the worst translations?
#    (e.g. wrong gender/number agreement, hallucinations, unknown words, reordering errors)
# 2. What do the best translations have in common?
# 3. Is there a relationship between sentence length and translation quality?
# 4. Suggest at least two concrete improvements to the model or training process
#    based on your findings.